<a href="https://colab.research.google.com/github/preetam7nemade-png/Compute-Task-1/blob/main/notebooks/01_eda_and_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn as sk

In [26]:
df=pd.read_csv('bank-additional-full.csv', sep=';')
df.shape

(41188, 21)

In [21]:
df.isnull().sum()

,0
age,0
job,0
marital,0
education,0
default,0
housing,0
loan,0
contact,0
month,0
day_of_week,0


In [22]:
df.describe()

,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
count,41188.00000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000
mean,40.02406,258.285010,2.567593,962.475454,0.172963,0.081886,93.575664,-40.502600,3.621291,5167.035911
std,10.42125,259.279249,2.770014,186.910907,0.494901,1.570960,0.578840,4.628198,1.734447,72.251528
min,17.00000,0.000000,1.000000,0.000000,0.000000,-3.400000,92.201000,-50.800000,0.634000,4963.600000
25%,32.00000,102.000000,1.000000,999.000000,0.000000,-1.800000,93.075000,-42.700000,1.344000,5099.100000
50%,38.00000,180.000000,2.000000,999.000000,0.000000,1.100000,93.749000,-41.800000,4.857000,5191.000000
75%,47.00000,319.000000,3.000000,999.000000,0.000000,1.400000,93.994000,-36.400000,4.961000,5228.100000
max,98.00000,4918.000000,56.000000,999.000000,7.000000,1.400000,94.767000,-26.900000,5.045000,5228.100000


In [23]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [24]:
print(f"Original dataset shape: {df.shape}")
target_col = 'y'
leakage_col = 'duration'

features = [col for col in df.columns if col not in [target_col, leakage_col]]
numeric_cols = df[features].select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df[features].select_dtypes(exclude=[np.number]).columns.tolist()
print(f"Numerical columns ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}\n")
print("Percentage of 'unknown' values per categorical column:")
for col in categorical_cols:
    unknown_count = (df[col] == 'unknown').sum()
    unknown_pct = (unknown_count / len(df)) * 100
    print(f"  {col}: {unknown_count} records ({unknown_pct:.2f}%)")
pdays_999_count = (df['pdays'] == 999).sum()
pdays_999_pct = (pdays_999_count / len(df)) * 100
print(f"\npdays = 999 (client was not previously contacted): {pdays_999_count} records ({pdays_999_pct:.2f}%)\n")
poutcome_y_ct = pd.crosstab(df['poutcome'], df['y'], normalize='index') * 100
print("Subscription rate (%) by previous outcome (poutcome):")
display(poutcome_y_ct)
emp_var_y_ct = pd.crosstab(df['emp.var.rate'], df['y'], normalize='index') * 100
print("\nSubscription rate (%) by employment variation rate (emp.var.rate):")
display(emp_var_y_ct)

Original dataset shape: (41188, 21)
Numerical columns (9): ['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
Categorical columns (10): ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']

Percentage of 'unknown' values per categorical column:
  job: 330 records (0.80%)
  marital: 80 records (0.19%)
  education: 1731 records (4.20%)
  default: 8597 records (20.87%)
  housing: 990 records (2.40%)
  loan: 990 records (2.40%)
  contact: 0 records (0.00%)
  month: 0 records (0.00%)
  day_of_week: 0 records (0.00%)
  poutcome: 0 records (0.00%)

pdays = 999 (client was not previously contacted): 39673 records (96.32%)

Subscription rate (%) by previous outcome (poutcome):


y,no,yes
poutcome,,
failure,85.771402,14.228598
nonexistent,91.167787,8.832213
success,34.887109,65.112891



Subscription rate (%) by employment variation rate (emp.var.rate):


y,no,yes
emp.var.rate,,
-3.4,57.609711,42.390289
-3.0,48.837209,51.162791
-2.9,64.281419,35.718581
-1.8,84.091899,15.908101
-1.7,47.865459,52.134541
-1.1,52.598425,47.401575
-0.2,90.000000,10.000000
-0.1,93.700787,6.299213
1.1,96.908412,3.091588
